> **Portfolio version.** Cell outputs and workspace-specific connection details have been removed. Configure the environment variables documented in the repository README before running on Databricks.


# Silver

Reads Bronze Delta tables, standardises borough keys, aligns time fields to UK financial years, and writes Silver tables.

Main analytical grain after this redesign is:

`borough + financial_year`

Static context sources (CSI and IMD) will be borough-level Silver tables and are joined later in Gold as contextual features, not annual features


In [ ]:
from datetime import datetime
from functools import reduce
import re

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType


## 1. Configuration

In [ ]:
catalog = ""
bronze_schema = "cbda_bronze"
silver_schema = "cbda_silver"
write_mode = "overwrite"

START_YEAR = 2011
END_YEAR = 2023
FINANCIAL_YEARS = [f"{year}-{str(year + 1)[-2:]}" for year in range(START_YEAR, END_YEAR + 1)]

VALID_BOROUGH_KEYS = [
    "barking and dagenham",
    "barnet",
    "bexley",
    "brent",
    "bromley",
    "camden",
    "croydon",
    "ealing",
    "enfield",
    "greenwich",
    "hackney",
    "hammersmith and fulham",
    "haringey",
    "harrow",
    "havering",
    "hillingdon",
    "hounslow",
    "islington",
    "kensington and chelsea",
    "kingston upon thames",
    "lambeth",
    "lewisham",
    "merton",
    "newham",
    "redbridge",
    "richmond upon thames",
    "southwark",
    "sutton",
    "tower hamlets",
    "waltham forest",
    "wandsworth",
    "westminster",
]

if catalog:
    bronze_ns = f"`{catalog}`.`{bronze_schema}`"
    silver_ns = f"`{catalog}`.`{silver_schema}`"
else:
    bronze_ns = f"`{bronze_schema}`"
    silver_ns = f"`{silver_schema}`"

print(f"Bronze namespace: {bronze_ns}")
print(f"Silver namespace: {silver_ns}")
print(f"Write mode: {write_mode}")
print(f"Financial years: {FINANCIAL_YEARS[0]} to {FINANCIAL_YEARS[-1]}")


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_ns}")


## 2. Helper Functions

In [ ]:
def table_name(namespace: str, table: str) -> str:
    return f"{namespace}.`{table}`"


def to_double(column_name: str):
    raw = F.trim(F.col(column_name).cast("string"))
    cleaned = F.regexp_replace(raw, ",", "")
    cleaned = F.regexp_replace(cleaned, "%", "")
    cleaned = F.regexp_replace(cleaned, "\\?", "")
    cleaned = F.regexp_replace(cleaned, "\\?", "")
    cleaned = F.regexp_replace(cleaned, "\\?", "")
    return (
        F.when(raw.isNull(), None)
        .when(F.lower(cleaned).isin("", "no data", "-", "#n/a", "#ref!", "nan", "null", ":", "!", "*", "#"), None)
        .when(~cleaned.rlike(r"^-?\d*\.?\d+$"), None)
        .otherwise(cleaned.cast("double"))
    )


def clean_key(column_name: str):
    text = F.lower(F.trim(F.regexp_replace(F.col(column_name).cast("string"), r"\s+", " ")))
    text = F.regexp_replace(text, "&", "and")
    text = F.regexp_replace(text, "-", " ")
    text = F.regexp_replace(text, r"\s+council$", "")
    text = F.regexp_replace(text, r"\s+", " ")
    return F.when(text.isin("", "unknown", "no data", "null"), None).otherwise(text)


def clean_display_name(column_name: str):
    text = F.trim(F.regexp_replace(F.col(column_name).cast("string"), r"\s+", " "))
    text = F.regexp_replace(text, "-", " ")
    lower = F.lower(text)
    return (
        F.when(lower == "bexley council", F.lit("Bexley"))
        .when(lower.isin("", "unknown", "no data", "null"), None)
        .otherwise(text)
    )


def valid_borough_filter():
    return F.col("borough_key").isin(VALID_BOROUGH_KEYS)


def financial_year_from_start_year(column_name: str):
    return F.concat(
        F.col(column_name).cast("string"),
        F.lit("-"),
        F.format_string("%02d", F.pmod(F.col(column_name) + F.lit(1), F.lit(100))),
    )


def stack_expr_for_columns(columns, value_alias="value_raw"):
    pairs = []
    for col in columns:
        label = col.replace("c_", "")
        pairs.append(f"'{label}', `{col}`")
    return f"stack({len(columns)}, {', '.join(pairs)}) as (stack_key, {value_alias})"


def save_delta(df, table: str):
    full_name = table_name(silver_ns, table)
    (
        df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    row_count = df.count()
    print(f"Wrote {full_name}: rows={row_count}, cols={len(df.columns)}")
    return row_count


def existing_columns(df, columns):
    return [c for c in columns if c in df.columns]

## 3. MPS Crime

In [ ]:
crime = spark.table(table_name(bronze_ns, "bronze_mps_crime"))

month_cols = sorted([c for c in crime.columns if re.match(r"^c_\d{6}$", c)])
if not month_cols:
    raise ValueError("No monthly crime columns found")

crime_long = (
    crime
    .select(
        "majortext",
        "minortext",
        "boroughname",
        F.expr(stack_expr_for_columns(month_cols, "crime_count_raw")),
    )
    .withColumnRenamed("stack_key", "year_month")
    .withColumn("borough_key", clean_key("boroughname"))
    .withColumn("borough_name", clean_display_name("boroughname"))
    .withColumn("calendar_year", F.substring("year_month", 1, 4).cast("int"))
    .withColumn("month", F.substring("year_month", 5, 2).cast("int"))
    .withColumn(
        "fy_start_year",
        F.when(F.col("month") >= 4, F.col("calendar_year")).otherwise(F.col("calendar_year") - F.lit(1)),
    )
    .withColumn("financial_year", financial_year_from_start_year("fy_start_year"))
    .withColumn("crime_count", F.coalesce(to_double("crime_count_raw"), F.lit(0.0)))
    .where(F.col("fy_start_year").between(START_YEAR, END_YEAR))
    .where(valid_borough_filter())
)

crime_silver = (
    crime_long
    .groupBy("borough_key", "borough_name", "financial_year", "fy_start_year")
    .agg(
        F.sum("crime_count").alias("crime_count"),
        F.count(F.lit(1)).alias("crime_month_category_records"),
        F.countDistinct("majortext").alias("major_crime_categories"),
        F.countDistinct("minortext").alias("minor_crime_categories"),
    )
)

save_delta(crime_silver, "silver_crime_borough_year")
display(crime_silver.orderBy("borough_name", "financial_year"))


## 4. Fly-Tipping 

In [ ]:
fly = spark.table(table_name(bronze_ns, "bronze_fly_tipping"))

fly_silver = (
    fly
    .withColumn("borough_key", clean_key("area"))
    .withColumn("borough_name", clean_display_name("area"))
    .withColumnRenamed("code", "borough_code")
    .withColumnRenamed("year", "financial_year")
    .withColumn("fy_start_year", F.substring("financial_year", 1, 4).cast("int"))
    .withColumn("flytipping_incidents", to_double("total_incidents"))
    .withColumn("flytipping_actions", to_double("total_action_taken"))
    .withColumn(
        "flytipping_enforcement_rate",
        F.when(F.col("flytipping_incidents") > 0, F.col("flytipping_actions") / F.col("flytipping_incidents")),
    )
    .where(F.col("fy_start_year").between(START_YEAR, END_YEAR))
    .where(valid_borough_filter())
    .select(
        "borough_code",
        "borough_key",
        "borough_name",
        "financial_year",
        "fy_start_year",
        "flytipping_incidents",
        "flytipping_actions",
        "flytipping_enforcement_rate",
    )
)

save_delta(fly_silver, "silver_flytipping_borough_year")
display(fly_silver.orderBy("borough_name", "financial_year"))


## 5. ONS Population

In [ ]:
population = spark.table(table_name(bronze_ns, "bronze_population"))

population_cols = sorted(
    [
        c for c in population.columns
        if re.match(r"^population_\d{4}$", c) and START_YEAR <= int(c.replace("population_", "")) <= END_YEAR
    ],
    key=lambda c: int(c.replace("population_", "")),
)
if not population_cols:
    raise ValueError("No population_YYYY columns found in bronze_population.")

population_stack = ", ".join([f"'{c.replace('population_', '')}', `{c}`" for c in population_cols])

population_silver = (
    population
    .select(
        F.col("ladcode23").alias("borough_code"),
        F.col("laname23").alias("borough_name_raw"),
        F.expr(f"stack({len(population_cols)}, {population_stack}) as (population_year, population_raw)"),
    )
    .withColumn("borough_key", clean_key("borough_name_raw"))
    .withColumn("borough_name", clean_display_name("borough_name_raw"))
    .withColumn("fy_start_year", F.col("population_year").cast("int"))
    .withColumn("financial_year", financial_year_from_start_year("fy_start_year"))
    .withColumn("population_mid_year", to_double("population_raw"))
    .where(valid_borough_filter())
    .select("borough_code", "borough_key", "borough_name", "financial_year", "fy_start_year", "population_mid_year")
)

save_delta(population_silver, "silver_population_borough_year")
display(population_silver.orderBy("borough_name", "financial_year"))


## 6. Income 

In [ ]:
income_raw = spark.table(table_name(bronze_ns, "bronze_income"))
income_cols = [c for c in income_raw.columns if not c.startswith("_")]

income_rowed = (
    income_raw
    .select(*income_cols)
    .withColumn("_row_id", F.row_number().over(Window.orderBy(F.monotonically_increasing_id())))
)

year_header = income_rowed.where(F.col("_row_id") == 1).first().asDict()
metric_header = income_rowed.where(F.col("_row_id") == 2).first().asDict()

current_year = None
year_metric_columns = {}
for col in income_cols:
    year_value = str(year_header.get(col) or "").strip()
    metric_value = str(metric_header.get(col) or "").strip().lower()

    if re.match(r"^\d{4}-\d{2}$", year_value):
        current_year = year_value

    if current_year in FINANCIAL_YEARS:
        if "number" in metric_value:
            year_metric_columns.setdefault(current_year, {})["income_taxpayer_count"] = col
        elif "mean" in metric_value:
            year_metric_columns.setdefault(current_year, {})["mean_income_gbp"] = col
        elif "median" in metric_value:
            year_metric_columns.setdefault(current_year, {})["median_income_gbp"] = col

missing_years = [fy for fy in FINANCIAL_YEARS if fy not in year_metric_columns]
if missing_years:
    raise ValueError(f"Income header parsing failed for financial years: {missing_years}")

income_data = income_rowed.where(F.col("_row_id") > 3)

income_frames = []
for fy in FINANCIAL_YEARS:
    cols = year_metric_columns[fy]
    fy_start_year = int(fy[:4])
    income_frames.append(
        income_data.select(
            F.col("c0").alias("borough_code"),
            F.col("c1").alias("borough_name_raw"),
            F.lit(fy).alias("financial_year"),
            F.lit(fy_start_year).alias("fy_start_year"),
            to_double(cols["income_taxpayer_count"]).alias("income_taxpayer_count"),
            to_double(cols["mean_income_gbp"]).alias("mean_income_gbp"),
            to_double(cols["median_income_gbp"]).alias("median_income_gbp"),
        )
    )

income_silver = reduce(lambda left, right: left.unionByName(right), income_frames)
income_silver = (
    income_silver
    .withColumn("borough_key", clean_key("borough_name_raw"))
    .withColumn("borough_name", clean_display_name("borough_name_raw"))
    .where(valid_borough_filter())
    .select(
        "borough_code",
        "borough_key",
        "borough_name",
        "financial_year",
        "fy_start_year",
        "income_taxpayer_count",
        "mean_income_gbp",
        "median_income_gbp",
    )
)

save_delta(income_silver, "silver_income_borough_year")
display(income_silver.orderBy("borough_name", "financial_year"))

## 7. Unemployment 

In [ ]:
unemployment = spark.table(table_name(bronze_ns, "bronze_unemployment"))

unemployment_number_cols = sorted([
    c for c in unemployment.columns
    if c.startswith("unemployment_rate_16") and c.endswith("_number")
])
unemployment_denominator_cols = sorted([
    c for c in unemployment.columns
    if c.startswith("unemployment_rate_16") and c.endswith("_denominator")
])

if not unemployment_number_cols or not unemployment_denominator_cols:
    print("Available unemployment columns:")
    print([c for c in unemployment.columns if "unemployment" in c])
    raise ValueError("Could not find unemployment number/denominator columns in bronze_unemployment.")

number_expr = reduce(
    lambda acc, c: acc + F.coalesce(to_double(c), F.lit(0.0)),
    unemployment_number_cols,
    F.lit(0.0),
)
denominator_expr = reduce(
    lambda acc, c: acc + F.coalesce(to_double(c), F.lit(0.0)),
    unemployment_denominator_cols,
    F.lit(0.0),
)
valid_expr = reduce(
    lambda acc, c: acc | to_double(c).isNotNull(),
    unemployment_number_cols,
    F.lit(False),
)

unemployment_base = (
    unemployment
    .withColumn("borough_key", clean_key("area"))
    .withColumn("borough_name", clean_display_name("area"))
    .withColumn("fy_start_year", F.col("year").cast("int"))
    .withColumn("unemployment_number_est_raw", F.when(valid_expr, number_expr))
    .withColumn("unemployment_denominator_est_raw", F.when(valid_expr, denominator_expr))
    .withColumn(
        "unemployment_rate_est_raw",
        F.when(
            (F.col("unemployment_denominator_est_raw") > 0) & F.col("unemployment_number_est_raw").isNotNull(),
            F.col("unemployment_number_est_raw") / F.col("unemployment_denominator_est_raw") * F.lit(100.0),
        ),
    )
    .where(F.col("fy_start_year").between(START_YEAR, END_YEAR))
    .where(valid_borough_filter())
)

unemployment_silver = (
    unemployment_base
    .groupBy("borough_key", "borough_name", "financial_year", "fy_start_year")
    .agg(
        F.sum("unemployment_number_est_raw").alias("unemployment_number_est_raw"),
        F.sum("unemployment_denominator_est_raw").alias("unemployment_denominator_est_raw"),
    )
    .withColumn(
        "unemployment_rate_est_raw",
        F.when(
            F.col("unemployment_denominator_est_raw") > 0,
            F.col("unemployment_number_est_raw") / F.col("unemployment_denominator_est_raw") * F.lit(100.0),
        ),
    )
)

# Silver keeps the raw estimate but also provides a simple continuity-filled estimate for Gold.
w_prev = Window.partitionBy("borough_key").orderBy("fy_start_year").rowsBetween(Window.unboundedPreceding, 0)
w_next = Window.partitionBy("borough_key").orderBy("fy_start_year").rowsBetween(0, Window.unboundedFollowing)

unemployment_silver = (
    unemployment_silver
    .withColumn("unemployment_rate_was_missing_raw", F.col("unemployment_rate_est_raw").isNull())
    .withColumn(
        "unemployment_rate_est",
        F.coalesce(
            F.col("unemployment_rate_est_raw"),
            F.last("unemployment_rate_est_raw", ignorenulls=True).over(w_prev),
            F.first("unemployment_rate_est_raw", ignorenulls=True).over(w_next),
        ),
    )
    .withColumn(
        "unemployment_rate_was_imputed",
        F.col("unemployment_rate_est_raw").isNull() & F.col("unemployment_rate_est").isNotNull(),
    )
)

save_delta(unemployment_silver, "silver_unemployment_borough_year")
display(unemployment_silver.orderBy("borough_name", "financial_year"))


## 8. CSI Combined Scores 

In [ ]:
csi = spark.table(table_name(bronze_ns, "bronze_csi"))

csi_score_cols = [c for c in csi.columns if c.startswith("csi_final_score")]
if not csi_score_cols:
    raise ValueError("No CSI Final Score columns found in bronze_csi.")

overall_csi_col = csi_score_cols[-1]
print(f"Using overall CSI score column: {overall_csi_col}")

csi_prepped = (
    csi
    .withColumn("borough_key", clean_key("la_name_2019_boundaries"))
    .withColumn("borough_name", clean_display_name("la_name_2019_boundaries"))
    .withColumn("ward_population", to_double("mye_population"))
    .withColumn("csi_final_score_num", to_double(overall_csi_col))
    .where(valid_borough_filter())
    .where(F.col("csi_final_score_num").isNotNull())
)

csi_silver = (
    csi_prepped
    .groupBy("borough_key", "borough_name")
    .agg(
        (
            F.sum(F.col("csi_final_score_num") * F.coalesce(F.col("ward_population"), F.lit(0.0)))
            / F.sum(F.coalesce(F.col("ward_population"), F.lit(0.0)))
        ).alias("csi_overall_score"),
        F.avg("csi_final_score_num").alias("csi_overall_score_unweighted"),
        F.countDistinct("ward_code_1").alias("csi_ward_count"),
        F.sum("ward_population").alias("csi_population_sum"),
    )
)

save_delta(csi_silver, "silver_csi_by_borough")
display(csi_silver.orderBy("borough_name"))


## 9. IMD 2019 

In [ ]:
imd = spark.table(table_name(bronze_ns, "bronze_imd"))

domain_candidates = [
    "income_average_score",
    "employment_average_score",
    "education_skills_and_training_average_score",
    "health_deprivation_and_disability_average_score",
    "crime_average_score",
    "barriers_to_housing_and_services_average_score",
    "living_environment_average_score",
]

domain_selects = [
    to_double(c).alias(c)
    for c in existing_columns(imd, domain_candidates)
]

imd_silver = (
    imd
    .withColumn("borough_key", clean_key("local_authority_district_name_2019"))
    .withColumn("borough_name", clean_display_name("local_authority_district_name_2019"))
    .select("borough_key", "borough_name", *domain_selects)
    .where(valid_borough_filter())
    .dropDuplicates(["borough_key"])
)

save_delta(imd_silver, "silver_imd_domains_by_borough")
display(imd_silver.orderBy("borough_name"))


## 10. Silver Audit

In [ ]:
silver_tables = [
    "silver_crime_borough_year",
    "silver_flytipping_borough_year",
    "silver_population_borough_year",
    "silver_income_borough_year",
    "silver_unemployment_borough_year",
    "silver_csi_by_borough",
    "silver_imd_domains_by_borough",
]

audit = []
for table in silver_tables:
    df = spark.table(table_name(silver_ns, table))
    audit.append((table, df.count(), len(df.columns)))

audit_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("column_count", LongType(), False),
])

audit_df = spark.createDataFrame(audit, audit_schema)
quality_table = table_name(silver_ns, "silver_quality_audit")
(
    audit_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quality_table)
)

display(audit_df.orderBy("table_name"))


In [ ]:
# Optional Silver export to ADLS Gen2
import os

STORAGE_ACCOUNT = os.environ.get("AZURE_STORAGE_ACCOUNT")
SILVER_CONTAINER = os.environ.get("AZURE_SILVER_CONTAINER", "silver")
SILVER_PREFIX = os.environ.get("AZURE_SILVER_PREFIX", "")

if not STORAGE_ACCOUNT:
    raise EnvironmentError("Set AZURE_STORAGE_ACCOUNT before exporting Silver tables.")

silver_adls_path = (
    f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{SILVER_PREFIX}"
    .rstrip("/") + "/"
)

def anon(text):
    return str(text).replace(STORAGE_ACCOUNT, "***")

silver_tables_list = [
    row.tableName
    for row in spark.sql(f"SHOW TABLES IN {silver_ns}").collect()
    if not row.tableName.startswith("_")
]

print(f"Exporting {len(silver_tables_list)} Silver tables to ADLS...")
for table in silver_tables_list:
    df = spark.table(table_name(silver_ns, table))
    output = f"{silver_adls_path}{table}"
    df.write.format("delta").mode("overwrite").save(output)
    print(f"  {table} -> {anon(output)}")

print(f"All Silver tables exported to {anon(silver_adls_path)}")
